# Lab 1: MDP Basics and (Offline) Planning
In this notebook, we implement basic notions in Markov decision processes (MDPs) and solve the MDP (offline) planning problem via dynamic programming.
We here focus on the finite-horizon setting.

In [ ]:
from abc import ABC, abstractmethod
import numpy as np

### Finite-horizon MDPs

An *finite-horizon* Markov Decision Process (MDP) is specified by tuple $M:=(\mathcal{S}, \mathcal{A}, P, R, H)$:
- State space $\mathcal{S}$.
- Action space $\mathcal{A}$.
- Transition function $P:\mathcal{S} \times \mathcal{A} \times [H] \to \Delta(\mathcal{S})$. When taking action $a$ in state $s$ at timestep $h$, the probability of transiting state $s'$ at the next timestep (i.e., $h+1$) is denoted as $P_h(s' | s, a )$.
- Reward function $R:\mathcal{S} \times \mathcal{A} \times [H] \to \mathbb{R}$. $R_h(s, a)$ is the immediate reward associated with taking action $a$ in state $s$ at timestep $h\in[H]$.
- Horizion $H$. A constant positive integer.

In each episode of this MDP, an initial state $s_1\in\mathcal{S}$ is drawn from some initial distribution $d_1\in\Delta(\mathcal{S})$. At each timestep $h \in[H]$, the agent observes state $s_h \in \mathcal{S}$, picks an action $a_h \in \mathcal{A}$, receives reward $r_h := R_h(s_h, a_h)$, and then the environment transitions to a next state $s_{h+1}$ drawn from distribution $P_h(\cdot | s_h, a_h)$. The episode ends when $s_{H+1}$ is reache at timestep $H+1$. This generates the following stochastic process:
$$
\tau = (s_1, a_1, r_1, s_2, a_2, r_2, ...,s_H, a_H, r_H, s_{H+1}).
$$
referred to as a trajectory.


In [ ]:
class FiniteHorizonMDP(ABC):
    ''' Finite-horizon MDP with finite state and action spaces'''
    @abstractmethod
    def get_state_space(self) -> list:
        """ 
        Returns the state space of this MDP
        Args: None
        Returns:
            - state_space: 
                an ordered list of all states
                state_space[i] is the i-th state; i is the state id/index
                 
        """
        pass

    @abstractmethod
    def get_action_space(self) -> list:
        """ 
        Returns the action space of this MDP
        Args: None
        Returns:
            - action_space:
                an ordered list of all actions
                action_space[i] is the i-th action; is the action id 
        """
        pass

    @abstractmethod
    def get_horizon(self):
        """ 
        Returns the horizon of this MDP
        Args: None
        Returns:
            - horizon H
        """
        pass

    @abstractmethod
    def get_reward(self, s, a, h):
        """ 
        Defines the (deterministic) reward function of this MDP
        Args:
            - s: state id,  0,1,...,nS-1
            - a: action id, 0,1,...,nA-1
            - h: timestep,  1,2,...,H
        Returns:
            - reward r_h(s,a)
        """
        pass
    
    @abstractmethod
    def get_transition(self, s, a, h) -> np.ndarray:
        """ 
        Defines the transition function of this MDP
        Args:
            - s: state id,  0,1,...,nS-1
            - a: action id, 0,1,...,nA-1
            - h: timestep,  1,2,...,H
        Returns:
            - next_s_prob:
                a np array of shape (nS,) 
                where nS is the number of states, 
                and the i-th item is the prob of transiting to i-th state 
                as ordered in method self.get_state_space
        """
        pass

A (Markov, stochastic) policy $\pi: \mathcal{S}\times[H] \to \Delta(\mathcal{A})$ selects action $a_h$ based on state $s_h$ at timestep $h$, written as $a_h\sim\pi_h(s_h)$, with $\pi_h(a_h|s_h)$ denoting the actual probability.
This policy defined above is *unstationary*, i.e., it depends on timestep $h$. 

In [ ]:
class UnstationaryPolicy(ABC):
    @abstractmethod
    def get_action_distribution(self, s, h) -> np.ndarray:
        """ 
        Args: 
            - s: state id, 0,1,...,nS-1
            - h: timestep, 1,2,...,H
        Returns:
            - a np array of shape (nA,) where nA is the number of actions; 
                the i-th item is the prob of selecting to i-th action 
        """
        pass

    @abstractmethod
    def sample_action(self, s, h):
        """ 
        Args: 
            - s: state id, 0,1,...,nS-1
            - h: timestep, 1,2,...,H
        Returns:
            - id of an action (0,1,...,nA-1) sampled from the distribution pi_h(s)
        """
        pass

In [ ]:
class RandomPolicy(UnstationaryPolicy):
    """ 
    A random policy that selects each action with equal probability
    """
    def __init__(self, action_space):
        self.action_space = action_space
        self.nA = len(action_space)

    def get_action_distribution(self, s, h):
        return np.ones(self.nA) / self.nA

    def sample_action(self, s, h):
        return np.random.choice(self.nA) # return action id

In [ ]:
class GreedyPolicy(UnstationaryPolicy):
    ''' 
    A greedy policy based on Q-values
    This policy selects the action with the highest Q-value for the given state and timestep.
    If there are multiple actions with the same maximum Q-value, it selects one of them uniformly at random.
    '''
    def __init__(self, Q, nA):
        self.Q = Q
        self.nA = nA
    
    def get_action_distribution(self, s, h):
        action_values = self.Q[h][s]
        max_value = np.max(action_values)
        distribution = np.zeros(self.nA)
        best_actions = np.where(action_values == max_value)[0]
        distribution[best_actions] = 1.0 / len(best_actions)
        return distribution
    
    def sample_action(self, s, h):
        return np.random.choice(self.nA, p=self.get_action_distribution(s, h))

The MDP tuple $M$, initial distribution $d_1$, and policy $\pi$ together fully determine the probablity of a certain trajectory $\tau$:
$$
\textstyle\Pr^{M, \pi}(\tau) = d_1(s_1) \cdot \pi_1(a_1 | s_1) \cdot P_1(s_2 | s_1, a_1) \cdot \pi_2(a_2 | s_2) \cdot P_2(s_3 | s_2, a_2) \cdots \pi_H(a_H | s_H) \cdot P_H(s_{H+1} | s_H, a_H)
.
$$
Here, the reward function does not appear because we assumed it is deterministic. 

The value functions are defined as
$$
V_h^{M, \pi}(s) := \mathbb{E}_{M,\pi}\left[\sum_{h^{\prime}=h}^H r_{h^{\prime}}\left(s_{h^{\prime}}, a_{h^{\prime}}\right) \mid s_h=s\right], \qquad
Q_h^{M, \pi}(s, a) := \mathbb{E}_{M,\pi}\left[\sum_{h^{\prime}=h}^H r_{h^{\prime}}\left(s_{h^{\prime}}, a_{h^{\prime}}\right) \mid s_h=s, a_h=a\right]
$$
where $\mathbb{E}_{M,\pi}$ means the expectation is with respect to following policy $\pi$ in MDP $M$.
We often omit $M$ in the notations above when the underlying MDP is clear from context.

In [ ]:
class SimulatorFiniteHorizonMDP():
    def __init__(self, mdp: FiniteHorizonMDP, policy: UnstationaryPolicy):
        self.mdp = mdp
        self.policy = policy

        self.state_space = mdp.get_state_space()
        self.action_space = mdp.get_action_space()
        self.H = mdp.get_horizon()
        self.nS = len(self.state_space)  # number of states
        self.nA = len(self.action_space)  # number of actions
        
    def step(self, s, a, h):
        """ 
        Samples a transition from taking action a in state s at timestep h
        Args:
            - s: state id,  0,1,...,nS-1
            - a: action id, 0,1,...,nA-1
            - h: timestep,  1,2,...,H
        Returns:
            - a sampled transition: a dict of following keys
            's': s
            'a': a
            'h': h
            'r': reward r_h(s,a) per reward function
            'next_s': next state sampled from the transition function
            'next_s_prob': the prob of the next state per transition function
        """

        ##### YOUR CODE: START  #################
        # Get reward
        r = self.mdp.get_reward(s, a, h)
        # Get transition probabilities
        next_s_prob = self.mdp.get_transition(s, a, h)
        # Sample next state according to the transition probabilities
        next_s = np.random.choice(self.nS, p=next_s_prob)
        
        # raise NotImplementedError("Fill me in")
        ##### YOUR CODE: END    ##################

        # Return the transition dictionary
        return {
            's': s,
            'a': a,
            'h': h,
            'r': r,
            'next_s': next_s,
            'next_s_prob': next_s_prob
        }
    
    def sample_trajectory(self, s, h, L = None):
        """ 
        Following self.policy, sample a trajectoryof length L starting from state s at timestep h
        If L is None, sample a trajectory until the end of the episode (i.e., horizon H)
        Args:
            - s: state id
            - h: timestep
            - L: length of the trajectory to sample; if None, set L properly
        Returns:
            - trajectory: a list of transitions, each is a dict as defined in method self.step
        """
        
        trajectory = []
        curr_s = s
        curr_h = h
        H = self.mdp.get_horizon()
        if L is None:
            L = H - h + 1  # number of steps until horizon

        for _ in range(L):
            # Sample action from policy
            a = self.policy.sample_action(curr_s, curr_h)
            # Take a step in the environment
            transition = self.step(curr_s, a, curr_h)
            trajectory.append(transition)
            # Prepare for next step
            curr_s = transition['next_s']
            curr_h += 1
        return trajectory
        
    def sample_trajectories(self, s, h, L = None, n = 1):
        """ 
        Sample n trajectories of length L starting from state s at timestep h
        If L is None, sample a trajectory until the end of the episode (i.e., horizon H)
        Args:
            - s: state id
            - h: timestep
            - L: length of the trajectory to sample; if None, set L properly
            - n: number of trajectories to sample
        Returns:
            - trajectories: a list of n trajectories, each is as defined in method self.sample_trajectory
        """
        
        trajectories = []
        for _ in range(n):
            traj = self.sample_trajectory(s, h, L)
            trajectories.append(traj)
        return trajectories
        
    def evaluate_policy(self, s, h, n = 1000):
        """ 
        Evaluate the policy starting from state s at timestep h
        Args:
            - s: state id
            - h: timestep
            - n: number of trajectories to sample for evaluation
        Returns:
            - estimated value V^pi_h(s): the average total reward of the n sampled trajectories
        """
        
        total_rewards = []
        for _ in range(n):
            traj = self.sample_trajectory(s, h)
            total_reward = sum([step['r'] for step in traj])
            total_rewards.append(total_reward)
        return np.mean(total_rewards)

### Planning via dynamic programming

By the principle of dynamic programming, one can show that the value functions satisfying the Bellman equation:
$$
V_h^\pi(s)=\textstyle\sum_{a \in \mathcal{A}} \pi_h(a | s) Q_h^\pi(s, a), \quad
Q_h^\pi(s, a)=r_h(s, a)+ \mathbb{E}_{s'\sim P_h(s,a)}\left[V^\pi_{h+1}(s')\right]
.
$$
We often write the matrix form of $\mathbb{E}_{s'\sim P_h(s,a)}\left[V^\pi_{h+1}(s')\right] = (P_h V_{h+1}^\pi)_{(s,a)}$, where $P_h V_{h+1}$ is a matrix multiplication with $P_h\in\mathbb{R}^{SA\times S}, V_{h+1}^\pi\in\mathbb{R}^{S}$ and the subscript $(s,a)$ denotes the $(s,a)$-th entry. 


It tunrs out that there is always a Markov policy $\pi^*$ that is optimal in the sense that $V^{\pi^*}_h(s) = \max_\pi V^\pi_h(s)$ and $Q^{\pi^*}_h(s,a) = \max_\pi Q^\pi_h(s,a)$ for all $(s,a,h)$. We write the optimal value function as $V^* := V^{\pi^*}, Q^*:=Q^{\pi^*}$. The optimal values satisfy the Bellman optimality equation:
$$
V_h^*(s)=\textstyle\max_{a \in \mathcal{A}} Q_h^*(s, a), \quad
Q_h^*(s, a)
= r_h(s, a)+ \mathbb{E}_{s'\sim P_h(s,a)}\left[V^*_{h+1}(s')\right]
= r_h(s, a)+ (P_h V_{h+1}^*)_{(s,a)}
.
$$
The policy greedy w.r.t. $Q^*$, $\pi^{Q^*}_h(s) := \argmax_a Q^*_h(s,a)$, is actually an optimal policy.

Another method to compute an optimal policy is *policy iteration*:
Starting with any policy $\pi^0$, for iteartions $k=1,2,\ldots$, We 1) compute the Q-value of policy $\pi^{k-1}$, $Q^{\pi^{k-1}}_h(s,a)$, for all $(s,a,h)$ and 2) update the policy as  $\pi^k_h(s):=\argmax_a Q^{\pi^{k-1}}_h(s,a)$ for all $(s,h)$.
Policy $\pi^k$ improves $\pi^{k-1}$ in the sense that:
For any $k \in[H]$, $\pi^k$ is optimal for any $h>H-k$, i.e.,
$Q_h^{\pi k}=Q_h^*, V_h^{\pi k}=V_h^*$  for all  $h>H-k$.

In [ ]:
class FiniteHorizonMDPPlanner:
    """ 
    Planning finite-horizon MDP by sovlving the Bellman and the Bellman optimal equations
    """
    def __init__(self, mdp: FiniteHorizonMDP):
        self.mdp = mdp
        self.H = mdp.get_horizon()
        self.state_space = mdp.get_state_space()
        self.action_space = mdp.get_action_space()
        self.nS = len(self.state_space)
        self.nA = len(self.action_space)
    
    def solve_bellman_equation(self, policy: UnstationaryPolicy):
        """ 
        Solve the Bellman equation for the given MDP and policy
        Returns:
            - V: a dict where V[h][s] is the value function at timestep h and state s
            - Q: a dict where Q[h][s][a] is the action-value function at timestep h, state s, and action a
        """
        V = {h: np.zeros(self.nS) for h in range(1, self.H + 2)} # V[H+1] is terminal value
        Q = {h: np.zeros((self.nS, self.nA)) for h in range(1, self.H + 1)}

        ##### YOUR CODE: START  #################
        for h in range(self.H, 0, -1):  # from H to 1
            for s in self.state_space:
                # Compute Q_h(s, a) for all actions a
                for a in self.action_space:
                    r = self.mdp.get_reward(s, a, h)
                    next_s_prob = self.mdp.get_transition(s, a, h)
                    next_value = np.dot(next_s_prob, V[h + 1])
                    Q[h][s][a] = r + next_value
                # Compute V_h(s)
                V[h][s] = np.sum(policy.get_action_distribution(s, h) * Q[h][s])
        # raise NotImplementedError("Fill me in")
        ##### YOUR CODE: END    ##################

        return V, Q
    
    def solve_bellman_optimality_equation(self):
        """ 
        Solve the Bellman optimality equation for the given MDP
        Returns:
            - V_star: a dict where V_star[h][s] is the optimal value function at timestep h and state s
            - Q_star: a dict where Q_star[h][s][a] is the optimal action-value function at timestep h, state s, and action a
        """
        V_star = {h: np.zeros(self.nS) for h in range(1, self.H + 2)}
        Q_star = {h: np.zeros((self.nS, self.nA)) for h in range(1, self.H + 1)}

        ##### YOUR CODE: START  #################
        for h in range(self.H, 0, -1):  # from H to 1
            for s in self.state_space:
                # Compute Q_star_h(s, a) for all actions a
                for a in self.action_space:
                    r = self.mdp.get_reward(s, a, h)
                    next_s_prob = self.mdp.get_transition(s, a, h)
                    next_value = np.dot(next_s_prob, V_star[h + 1])
                    Q_star[h][s][a] = r + next_value
                # Compute V_star_h(s)
                V_star[h][s] = np.max(Q_star[h][s])
        # raise NotImplementedError("Fill me in")
        ### YOUR CODE: END    ##################
        
        return V_star, Q_star

    def extract_greedy_policy(self, Q):
        """ 
        Extract the greedy policy from the given action-value function Q
        Args:
            - Q: a dict where Q[h][s][a] is the action-value function at timestep h, state s, and action a
        Returns:
            - policy: an instance of UnstationaryPolicy representing the greedy policy
        """
        return GreedyPolicy(Q, self.nA)
    
    def policy_iteration(self, initial_policy: UnstationaryPolicy):
        """ 
        Perform policy iteration to find the optimal policy
        Args:
            - initial_policy: an instance of UnstationaryPolicy representing the initial policy
        Returns:
            - policy: an instance of UnstationaryPolicy representing the optimal policy
            - V: a dict where V[h][s] is the value function at timestep h and state s for the optimal policy
            - Q: a dict where Q[h][s][a] is the action-value function at timestep h, state s, and action a for the optimal policy
        """
        policy = initial_policy
        for _ in range(self.H):
            V, Q = self.solve_bellman_equation(policy)
            policy = self.extract_greedy_policy(Q)
        return policy, V, Q


### Example: Combination Lock

Consider the MDP depicted below, with $H+2$ states and two actions. The "good" green action deterministically leads to the next state in the chain, while the "bad" red action deterministically leads to a terminal state.
The reward does not depend on timestep and is annotated for each each state-action pair.

<img src="combination_lock.png" alt="combination_lock" width="700"/>


In [ ]:
class FiniteHorizonCombinationLock(FiniteHorizonMDP):
    """ 
    A finite-horizon MDP with a combination lock structure
    """
    def __init__(self, H):
        self.H = H
        self.state_space = list(range(H + 2)) # states: 1 to H+1 are the chain states; 0 is the terminal state
        self.action_space = [0, 1]            # actions: 0 (good), 1 (bad)

    def get_state_space(self):
        return self.state_space

    def get_action_space(self):
        return self.action_space

    def get_horizon(self):
        return self.H

    def get_reward(self, s, a, h):
        H = self.H
        if s == 0 or s == H+1:  # terminal state
            return 0.
        elif s == H and a == 0: # good action end of chain
            return H + 10.
        elif a == 0:            # good action mid of chain
            return -1.
        else:                   # bad action mid of chain
            return 1.           

    def get_transition(self, s, a, h):
        next_s_prob = np.zeros(len(self.state_space))

        ##### YOUR CODE: START  #################
        if a == 0:  # good action
            if s == 0 or s == self.H + 1:  # terminal state or beyond
                next_s_prob[0] = 1.0
            else:  # valid state in the chain
                next_s_prob[s + 1] = 1.0
        else:  # bad action
            next_s_prob[0] = 1.0  # transition to terminal state 0
        
        # raise NotImplementedError("Fill me in")
        ### YOUR CODE: END    ##################

        return next_s_prob

Let's evaluate the policy that selects actions uniformaly at random in Combination Lock.

In [ ]:
# Let's evaluate the policy that selects actions uniformaly at random in Combination Lock.
H = 5
mdp = FiniteHorizonCombinationLock(H=H)
random_policy = RandomPolicy(mdp.get_action_space())

We first evaluate the random policy by simulate multiple trjaectories and average their episodic rewards.
This is implmented as the `evaluate_policy` method in the `SimulatorFiniteHorizonMDP` class. 

In [ ]:
s, h = 1, 1  # starting state and timestep
simulator = SimulatorFiniteHorizonMDP(mdp, random_policy)
v_pi_s_h = simulator.evaluate_policy(s, h, n=10000)
print(f"Estimated value V^pi_{h}({s}) = {v_pi_s_h:.4f}")

Solving the Bellman equation gives the exact values of $V^\pi$, which is implemented in the `solve_bellman_equation` method in the `FiniteHorizonMDPPlanner` class.

In [ ]:
planner = FiniteHorizonMDPPlanner(mdp)
V, Q = planner.solve_bellman_equation(random_policy)
print(f"Exact value V^pi_{h}({s}) = {V[h][s]:.4f}")

We next solve the Bellman optimality equation for an optimal policy.

In [ ]:
V_star, Q_star = planner.solve_bellman_optimality_equation()
print(f"Exact optimal value V*_{h}({s}) = {V_star[h][s]:.4f}")
for a in range(planner.nA):
    print(f"Exact optimal value Q*_{h}({s},{a}) = {Q_star[h][s][a]:.4f}")

# Extract the greedy policy from the optimal Q-values
optimal_policy = planner.extract_greedy_policy(Q_star)
print(f"Optimal policy action distribution at state {s}, timestep {h}: {optimal_policy.get_action_distribution(s, h)}")
# Estimate the value of the optimal policy
simulator.policy = optimal_policy
v_star_s_h = simulator.evaluate_policy(s, h, n=1000)
print(f"Estimated value V*_{h}({s}) = {v_star_s_h:.4f}")